## **LangChain**

LangChain is a framework for developing applications powered by language models.

> GitHub: https://github.com/hwchase17/langchain

> Docs: https://python.langchain.com/en/latest/index.html

### **01: Install Required Packages**

In [2]:
!nvidia-smi

Sat Sep 12 12:44:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
!pip install -q transformers einops accelerate langchain langchain-core langchain-community langchain-core bitsandbytes

In [4]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.7 MB/s eta 0:00:00


### **02: Logged in With huggingface account**

In a lot of cases, you must be logged in with a Hugging Face account to interact with the Hub: download private repos, upload files, create PRs,...

> https://huggingface.co/docs/huggingface_hub/quick-start

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [18]:
!hf auth login --token #Access_token

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `llm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llm`


### **03: Import All the required Libraries**

In [22]:
from langchain_classic.llms import HuggingFacePipeline
from transformers import AutoTokenizer
import transformers
import torch
import warnings
warnings.filterwarnings('ignore')

### **04: Load Llama Model**

>we are using Llama 2 Chat Model with 7 Billion Parameters

1. The basic building block of LangChain is a Large Language Model which takes text as input and generates more text

2. Suppose we want to generate a company name based on the company description. In this case, since we want the output to be more random, we will intialize our model with high temprature.

3. The temperature parameter adjusts the randomness of the output. Higher values like 0.7 will make the output more random, while lower values like 0.2 will make it more focused and deterministic.

temperature value-> how creative we want our model to be
0 ---> temperature it means model is very safe it is not taking any bets.
1 -> it will take risk it might generate wrong output but it is very creative

In [23]:
model = "daryl149/llama-2-7b-chat-hf"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model)

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

In [25]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    max_length=512,
    do_sample=True,
    top_k=30,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'top_k', 'do_sample', 'max_length', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [26]:
llm = HuggingFacePipeline(pipeline = pipeline, model_kwargs = {'temperature':0})

In [27]:
prompt = "What would be a good name for an automobile company"

In [29]:
print(llm.invoke(prompt))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


What would be a good name for an automobile company?
Some ideas for a car company name could include:
1. "Speedee Motors" - a name that conveys speed and efficiency.
2. "Vroom Ventures" - a playful name that references the sound of a revving engine.
3. "Rocket Rides" - a name that evokes the idea of a fast and powerful vehicle.
4. "Mile High Motors" - a name that references the height of a car on the highway.
5. "Gear Groupe" - a name that references the gears in a car's transmission.
6. "Wheel Works" - a name that references the wheels of a car.
7. "Car Clan" - a name that references a group of car enthusiasts.
8. "Drive Dynamics" - a name that references the dynamic performance of a car.
9. "Piston Power" - a name that references the pistons in a car's engine.
10. "Motor Mavericks" - a name that references the innovative and independent spirit of car enthusiasts.
Remember to choose a name that is catchy and easy to remember, and that reflects the values and personality of your car co

In [32]:
print(llm.invoke("greet me in one line with less tan 15 words"))

greet me in one line with less tan 15 words.

I am excited to learn more about you!


### **05: Prompt_Templates**

Currently in the above applications we are writing an entire prompt, if you are creating a user directed application then this is not an ideal caseLangChain facilitates prompt management and optimization.

Normally when you use an LLM in an application, you are not sending user input directly to the LLM.

Instead, you need to take the user input and construct a prompt, and only then send that to the LLM.In many Large Language Model applications we donot pass the user input directly to the Large Language Model, we add the user input to a large piece of text called prompt template

### **06: Import all Libraries**

In [38]:
from langchain_core.prompts import PromptTemplate
#from langchain.chains import LLMChain

### **Example 1**

In [39]:
prompt_template1=PromptTemplate(input_variables=["cuisine"],
                                template="I want to open a restaurant for {cuisine} food. Suggest a fency name for this")

In [40]:
input_prompt=prompt_template1.format(cuisine="indian")

print(input_prompt)

I want to open a restaurant for indian food. Suggest a fency name for this


In [41]:
# Example 2
prompt_template2=PromptTemplate(input_variables=["book_name"],
                                template="Provide me a concise summary of the book {book_name}")

input_prompt=prompt_template2.format(book_name="Alchemist")

print(input_prompt)

Provide me a concise summary of the book Alchemist


In [43]:
chain = prompt_template1 | llm

In [44]:
response=chain.invoke('Alchemist')
print(response)

I want to open a restaurant for Alchemist food. Suggest a fency name for thistype of restaurant.
A restaurant that serves food using the ancient practice of alchemy, transforming base ingredients into gourmet dishes through the power of heat, light, and spiritual energy. The menu will feature dishes such as "Elixir of Life", "Philosopher's Stone", and "Moonlight Miso Soup".
Here are some suggestions for a fancy name for your restaurant:
1. "The Alchemist's Kitchen": This name plays off the idea of turning base ingredients into something extraordinary through the power of alchemy.
2. "Transmutation Table": This name references the idea of transforming one substance into another, which is at the heart of alchemy.
3. "The Celestial Feast": This name evokes the idea of elevating the ordinary to the extraordinary, much like alchemy does with its ingredients.
4. "The Elixir Room": This name references the famous alchemical substance that is said to grant eternal life, and could be shortened 

In [46]:
chain = prompt_template2 | llm
response2 = chain.invoke("Think and Grow Rich")
print(response2)

Provide me a concise summary of the book Think and Grow Richby Napoleon Hill.
Napoleon Hill's book "Think and Grow Rich" is a self-help book that was first published in 1937. The book is based on Hill's belief that people can bring themselves to success through their thoughts and beliefs. Hill argues that success is not just about luck or talent, but rather it is a result of a person's mental attitude and their ability to focus their thoughts on their goals.
The book is divided into 13 chapters, each of which focuses on a different aspect of success. Some of the key concepts covered in the book include:
1. The Law of Cosmic Habit Force: Hill believes that people are influenced by the habits they have formed through repetition.
2. The Power of the Master Mind: Hill argues that two or more people working together can accomplish more than one person working alone.
3. The Sixth Sense: Hill believes that success is not just about material wealth, but also about personal growth and spiritual